# Detector
Notebook version of `Service/detector.py`. It loads `common.ipynb` and `muse_client.ipynb` automatically.

In [ ]:
from pathlib import Path
NB_DIR = Path.cwd() if Path.cwd().name == "Service_jupyter" else Path.cwd() / "Service_jupyter"
get_ipython().run_line_magic("run", str(NB_DIR / "common.ipynb"))
get_ipython().run_line_magic("run", str(NB_DIR / "muse_client.ipynb"))

In [ ]:
import threading
import time
from typing import Any

import numpy as np
from scipy import signal

MUSE_CHANNEL_ORDER_FOR_MODEL = ["TP9", "TP10", "AF7", "AF8"]

def replace_bad_values(window: np.ndarray) -> np.ndarray:
    cleaned = window.copy()
    for channel_index in range(cleaned.shape[0]):
        channel = cleaned[channel_index]
        good = np.isfinite(channel)
        channel[~good] = float(np.median(channel[good])) if good.any() else 0.0
    return cleaned

def bandpass_filter(window: np.ndarray, sampling_rate: int) -> np.ndarray:
    nyquist = sampling_rate / 2.0
    sos = signal.butter(
        FILTER_ORDER,
        [FILTER_LOW_HZ / nyquist, FILTER_HIGH_HZ / nyquist],
        btype="bandpass",
        output="sos",
    )
    return signal.sosfiltfilt(sos, window, axis=1)

def probability_map(model, probabilities, label_names):
    if probabilities is None:
        return None
    classes = getattr(model, "classes_", None)
    if classes is None and hasattr(model, "named_steps"):
        classes = getattr(list(model.named_steps.values())[-1], "classes_", None)
    if classes is None:
        classes = np.arange(len(probabilities))
    return {label_names.get(int(c), str(c)): float(probabilities[i]) for i, c in enumerate(classes)}

class EmotionDetectionService:
    def __init__(self, receiver: MuseReceiver, model_file: Path = TRAINED_MODEL_FILE, step_seconds: float = 1.0):
        self.receiver = receiver
        self.model_file = model_file
        self.step_seconds = step_seconds
        self.window_seconds = WINDOW_SECONDS
        self.model_sample_rate = SAMPLING_RATE
        self.muse_samples_needed = int(self.window_seconds * MUSE_SAMPLE_RATE)
        self.model_samples_needed = int(self.window_seconds * self.model_sample_rate)
        self._lock = threading.Lock()
        self._thread = None
        self._stop_event = threading.Event()
        self._artifact = None
        self.running = False
        self.last_error = ""
        self.latest_result = None
        self.history = []

    def start(self):
        if self._thread and self._thread.is_alive():
            return
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_event.set()
        if self._thread:
            self._thread.join(timeout=3)

    def _load_artifact_once(self):
        if self._artifact is None:
            self._artifact = load_model(self.model_file)
        return self._artifact

    def _run(self):
        self.running = True
        self.last_error = ""
        try:
            self._load_artifact_once()
            while not self._stop_event.is_set():
                raw_window = self.receiver.latest_window(self.muse_samples_needed, MUSE_CHANNEL_ORDER_FOR_MODEL)
                if raw_window is not None:
                    result = self.predict(raw_window)
                    with self._lock:
                        self.latest_result = result
                        self.history.append(result)
                        self.history = self.history[-20:]
                time.sleep(self.step_seconds)
        except Exception as error:
            self.last_error = str(error)
        finally:
            self.running = False

    def preprocess_live_window(self, muse_window):
        window = np.asarray(muse_window, dtype=np.float64)
        window = replace_bad_values(window)
        window = signal.resample(window, self.model_samples_needed, axis=1)
        window = signal.detrend(window, axis=1, type="constant")
        window = bandpass_filter(window, self.model_sample_rate)
        return window.astype(np.float32)

    def predict(self, muse_window):
        artifact = self._load_artifact_once()
        model = artifact["model"]
        label_names = {int(k): v for k, v in artifact.get("label_names", LABEL_NAMES).items()}
        model_window = self.preprocess_live_window(muse_window)
        features, _ = extract_feature_vector(
            model_window,
            sampling_rate=self.model_sample_rate,
            channel_names=SELECTED_SEED_CHANNELS,
        )
        features = features.reshape(1, -1)
        predicted_class = int(model.predict(features)[0])
        probabilities = model.predict_proba(features)[0] if hasattr(model, "predict_proba") else None
        return {
            "timestamp": time.time(),
            "predicted_class": predicted_class,
            "predicted_label": label_names.get(predicted_class, str(predicted_class)),
            "probabilities": probability_map(model, probabilities, label_names),
            "model_name": artifact.get("model_name", "unknown"),
            "window_seconds": self.window_seconds,
        }

    def status(self):
        with self._lock:
            latest = dict(self.latest_result) if self.latest_result else None
            history = list(self.history)
        return {
            "running": self.running,
            "last_error": self.last_error,
            "model_file": str(self.model_file),
            "window_seconds": self.window_seconds,
            "muse_samples_needed": self.muse_samples_needed,
            "latest_result": latest,
            "history": history,
        }

print("Detector definitions loaded.")